# WavLM Real Audio Training — Colab GPU v6 (FIXED)

Trains on REAL audio from GDrive on GPU.
Target: F1 >= 0.55

**FIXES from v5:**
- Learning rate: 1e-4 (was 1e-3 — too high for frozen encoder)
- Added CosineAnnealing LR scheduler
- Added gradient clipping (max_norm=1.0)

**Runtime → Change runtime type → GPU → Run All**

In [ ]:
# Setup
!pip install -q transformers librosa soundfile accelerate

import torch
import numpy as np
import json
import time
import os
import librosa
from collections import Counter
from math import cos
from transformers import WavLMModel
from sklearn.metrics import f1_score, precision_score, recall_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU!")

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/gdrive', force_remount=True)
print("GDrive mounted!")

In [ ]:
# Build video_id → audio path mapping using os.listdir
BASE = "/content/gdrive/MyDrive/laughter_prediction"

def list_gdrive_videos(folder):
    folder_path = f"{BASE}/audio/{folder}"
    video_map = {}
    if os.path.exists(folder_path):
        for fname in os.listdir(folder_path):
            if fname.endswith('.mp3'):
                vid = fname.replace('.mp3', '')
                video_map[vid] = f"{folder}/{fname}"
    return video_map

print("Scanning GDrive audio folders...")
en_map = list_gdrive_videos("en")
print(f"  en: {len(en_map)} videos")
zh_map = list_gdrive_videos("zh")
print(f"  zh: {len(zh_map)} videos")
hi_map = list_gdrive_videos("hi-latn")
print(f"  hi-latn: {len(hi_map)} videos")

GDrive_AUDIO_MAP = {}
GDrive_AUDIO_MAP.update(en_map)
GDrive_AUDIO_MAP.update(zh_map)
GDrive_AUDIO_MAP.update(hi_map)
print(f"Total GDrive audio videos: {len(GDrive_AUDIO_MAP)}")

In [ ]:
# Load segments from batch1 and batch2
MAX_SAMPLES = 80000
SR = 16000
samples = []

def load_segments(filepath, max_lines):
    local_samples = []
    try:
        with open(filepath) as f:
            for i, line in enumerate(f):
                if i >= max_lines:
                    break
                try:
                    d = json.loads(line)
                    vid = d.get('video_id', '')
                    start = d.get('start')
                    end = d.get('end')
                    label = int(d.get('label', 0)) if str(d.get('label', '0')).isdigit() else 0
                    
                    if vid in GDrive_AUDIO_MAP:
                        full_path = f"{BASE}/audio/{GDrive_AUDIO_MAP[vid]}"
                        if os.path.exists(full_path) and start is not None and end is not None and end > start:
                            local_samples.append({
                                "audio_file": full_path,
                                "start": start,
                                "end": end,
                                "label": label
                            })
                except:
                    pass
    except FileNotFoundError:
        print(f"  File not found: {filepath}")
    return local_samples

print("Loading batch1 segments...")
batch1 = load_segments(f"{BASE}/aligned/batch1/aligned_segments_batch1.jsonl", MAX_SAMPLES)
print(f"  Loaded {len(batch1)} segments with GDrive audio")
print("Loading batch2 segments...")
batch2 = load_segments(f"{BASE}/aligned/batch2/aligned_segments_batch2.jsonl", MAX_SAMPLES)
print(f"  Loaded {len(batch2)} segments with GDrive audio")

samples = batch1 + batch2
print(f"\nTotal: {len(samples)} segments with audio")

labels = Counter(s["label"] for s in samples)
print(f"Laugh: {labels[1]} ({100*labels[1]/len(samples):.1f}%)")
print(f"No laugh: {labels[0]} ({100*labels[0]/len(samples):.1f}%)")

In [ ]:
# Audio loading with LRU cache
from functools import lru_cache

@lru_cache(maxsize=32)
def load_audio_cached(path):
    try:
        y, _ = librosa.load(path, sr=SR, mono=True)
        return y
    except Exception as e:
        return None

def extract_segment(path, start, end, pad_ms=50):
    y = load_audio_cached(path)
    if y is None:
        return np.zeros(int(0.3 * SR), dtype=np.float32)
    s = int(max(0, start - pad_ms/1000) * SR)
    e = int(min(len(y)/SR, end + pad_ms/1000) * SR)
    if s >= len(y) or e <= s:
        return np.zeros(int(0.3 * SR), dtype=np.float32)
    seg = y[s:e]
    if len(seg) < int(0.1 * SR):
        seg = np.pad(seg, (0, int(0.1 * SR) - len(seg)))
    return seg.astype(np.float32)

# Test audio loading
test = extract_segment(samples[0]["audio_file"], samples[0]["start"], samples[0]["end"])
print(f"Audio test: shape={test.shape}, duration={len(test)/SR:.3f}s")

In [ ]:
# Load WavLM-Base+
print("Loading WavLM-Base+ (94M params)...")
t0 = time.time()
encoder = WavLMModel.from_pretrained("microsoft/wavlm-base-plus")
encoder.to(device)
encoder.eval()
print(f"Loaded in {time.time()-t0:.1f}s")

In [ ]:
# Model: frozen WavLM + MLP head
class WavLMLaughterClassifier(torch.nn.Module):
    def __init__(self, enc):
        super().__init__()
        self.encoder = enc
        for p in self.encoder.parameters():
            p.requires_grad = False
        self.classifier = torch.nn.Sequential(
            torch.nn.Linear(768, 256), torch.nn.ReLU(), torch.nn.Dropout(0.2),
            torch.nn.Linear(256, 64), torch.nn.ReLU(), torch.nn.Dropout(0.2),
            torch.nn.Linear(64, 2)
        )
    def forward(self, x):
        with torch.no_grad():
            h = self.encoder(x).last_hidden_state.mean(dim=1)
        return self.classifier(h)

model = WavLMLaughterClassifier(encoder).to(device)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params: {trainable:,}")

In [ ]:
# Training with FIXED hyperparameters
BATCH, EPOCHS = 64, 5
LR = 1e-4  # FIXED: was 1e-3, too high for frozen encoder

optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)
criterion = torch.nn.CrossEntropyLoss(weight=torch.tensor([1., 5.]).to(device))

# FIXED: CosineAnnealing scheduler for smooth LR decay
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=len(samples) // BATCH * EPOCHS)

print(f"Training: {EPOCHS} epochs, batch={BATCH}, LR={LR}, samples={len(samples)}")
print(f"Device: {device}")
print(f"Fixes: LR=1e-4 (was 1e-3), CosineAnnealing, gradient clipping")

for ep in range(1, EPOCHS+1):
    t0 = time.time()
    preds, ys = [], []
    nb = (len(samples) + BATCH - 1) // BATCH
    model.train()
    
    for bi in range(nb):
        batch = samples[bi*BATCH : (bi+1)*BATCH]
        
        audio_batch = [extract_segment(s["audio_file"], s["start"], s["end"]) for s in batch]
        max_len = max(len(a) for a in audio_batch)
        padded = np.zeros((len(audio_batch), max_len), dtype=np.float32)
        for i, a in enumerate(audio_batch):
            padded[i, :len(a)] = a
        
        x = torch.from_numpy(padded).to(device)
        y = torch.tensor([s["label"] for s in batch]).to(device)
        
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        
        # FIXED: Gradient clipping to prevent exploding gradients
        torch.nn.utils.clip_grad_norm_(filter(lambda p: p.requires_grad, model.parameters()), max_norm=1.0)
        
        optimizer.step()
        scheduler.step()
        
        preds.extend(model(x).argmax(1).cpu().tolist())
        ys.extend(y.cpu().tolist())
        
        if (bi+1) % 100 == 0:
            current_lr = scheduler.get_last_lr()[0]
            print(f"  Batch {bi+1}/{nb}, loss={loss.item():.4f}, lr={current_lr:.2e}")
    
    f1 = f1_score(ys, preds, zero_division=0)
    p = precision_score(ys, preds, zero_division=0)
    r = recall_score(ys, preds, zero_division=0)
    print(f"Epoch {ep}/{EPOCHS} ({time.time()-t0:.0f}s) F1={f1:.4f} P={p:.4f} R={r:.4f}")

In [ ]:
# Save to GDrive
save_path = "/content/gdrive/MyDrive/wavlm_phaseA_colab_v6.pt"
torch.save(model.state_dict(), save_path)
print(f"Saved: {save_path}")